# 2-3. Retriever 패턴: similarity / MMR / threshold / 메타필터 / 멀티테넌트

⏱ **소요시간**: 30분

## 학습 목표
- similarity / MMR / threshold 기반 Retriever의 차이를 구분한다.
- 메타데이터(소스, 작성일, 부서 등)를 활용한 필터링을 구현할 수 있다.
- 금융권 **멀티테넌트 격리**를 `tenant_id` 필터로 구현한다.
- Time-weighted Retriever의 동작 원리를 설명하고 간단히 시연한다.

## 핵심 키워드
`similarity` `MMR` `score_threshold` `metadata filter` `multi-tenant` `tenant_id` `time-weighted` `as_retriever`

> 🔒 모든 리트리버는 로컬 Chroma 에 구축된 인덱스에서 동작하여 폐쇄망 친화적이다.

In [ ]:
import sys; sys.path.insert(0, '../..')
from common import get_llm, get_embeddings, provider_badge
print(provider_badge())

In [ ]:
# 실습용 코퍼스: 금융 도메인의 다중 테넌트 문서를 모상
from datetime import datetime, timedelta
from langchain_core.documents import Document

now = datetime(2026, 4, 17)
D = lambda d, days=0, **m: Document(page_content=d, metadata={'date': (now - timedelta(days=days)).date().isoformat(), **m})

docs = [
    # tenant A — 리테일 은행 (B2C)
    D('결제카드 분실시 고객센터 1588-0000으로 즉시 신고하여야 합니다.', days=10, tenant_id='retail_bank', source='cs_manual', topic='lost_card'),
    D('청약철회는 계약일로부터 14일 이내에 서면으로 가능합니다.', days=120, tenant_id='retail_bank', source='terms', topic='withdrawal'),
    D('2026년 1월부터 오픈뱅킹 수수료가 무료로 전환됩니다.', days=90, tenant_id='retail_bank', source='announce', topic='fee'),
    # tenant B — 투자증권 (IB)
    D('기관투자자 전용 프라임 브로커리지 수수료는 거래액의 0.03%입니다.', days=5, tenant_id='securities', source='ib_terms', topic='fee'),
    D('ELS 상품의 중도상환 수수료는 잔존기간에 따라 차등 적용됩니다.', days=60, tenant_id='securities', source='product', topic='product'),
    # tenant C — 보험
    D('자동차보험 신규 가입은 면허 보유 1년 경과 시 우대 적용 가능.', days=15, tenant_id='insurance', source='policy', topic='auto'),
    D('실속 의료보험은 보장 제한 사항을 약관 제5조에서 확인하세요.', days=400, tenant_id='insurance', source='terms', topic='medical'),
]
print(f'총 {len(docs)}개 문서, 테넌트 종류: {sorted({d.metadata["tenant_id"] for d in docs})}')

In [ ]:
import shutil
from pathlib import Path
from langchain_community.vectorstores import Chroma

CHROMA_DIR = Path('./_store/chroma_retriever')
if CHROMA_DIR.exists():
    shutil.rmtree(CHROMA_DIR)
CHROMA_DIR.parent.mkdir(parents=True, exist_ok=True)

vs = Chroma.from_documents(docs, embedding=get_embeddings(), persist_directory=str(CHROMA_DIR), collection_name='fin_multi_tenant')
print('인덱싱된 문서:', vs._collection.count())

## 1. similarity vs MMR vs threshold

In [ ]:
q = '카드 분실 신고 방법 알려줘'

sim_retriever = vs.as_retriever(search_type='similarity', search_kwargs={'k': 3})
for d in sim_retriever.invoke(q):
    print(f'[similarity] {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

In [ ]:
# MMR (Maximal Marginal Relevance) — 관련성 + 다양성 균형
# fetch_k: 최종적으로 k개 고르기 위해 먼저 가져올 후보 갯수
# lambda_mult: 1에 가까울수록 엄격한 유사도, 0에 가까울수록 다양성 상승
mmr_retriever = vs.as_retriever(search_type='mmr', search_kwargs={'k': 3, 'fetch_k': 10, 'lambda_mult': 0.3})
for d in mmr_retriever.invoke(q):
    print(f'[mmr]        {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

In [ ]:
# score threshold — 유사도가 일정 이상인 결과만 반환 (hallucination 방지에 유용)
thr_retriever = vs.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k': 5, 'score_threshold': 0.4})
hits = thr_retriever.invoke(q)
print(f'[threshold=0.4] hits={len(hits)}')
for d in hits:
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["topic"]}: {d.page_content[:50]}…')

# 임계값을 높이면 결과가 0개가 될 수 있다 → "못찾았습니다" 처리로 연결
thr2 = vs.as_retriever(search_type='similarity_score_threshold', search_kwargs={'k': 5, 'score_threshold': 0.85})
print(f'[threshold=0.85] hits={len(thr2.invoke(q))}  ← 텍스트 없으면 LLM이 "모르겠습니다" 로 답하도록 설정')

## 2. 메타데이터 필터링

Chroma는 `$and / $or / $eq / $gte` 등을 지원한다.

In [ ]:
# source == 'terms' 인 문서만 필터링
# Chroma의 $gte는 숫자형에만 적용 가능하므로, 문자열 date는 $eq 또는 Python 후처리로 필터한다.
recent_terms = vs.as_retriever(search_kwargs={
    'k': 5,
    'filter': {'source': {'$eq': 'terms'}},
})
for d in recent_terms.invoke('약관 관련 조항'):
    print(f'  date={d.metadata["date"]} source={d.metadata["source"]}: {d.page_content[:50]}…')

## 3. 멀티테넌트 격리 패턴

리테일 은행 직원이 투자증권/보험 사업부 문서를 조회하면 안 되는 상황 — **동일 벡터 DB에 여러 테넌트를 두고 필터로 분리**하는 것이 표준 패턴.

In [ ]:
def tenant_retriever(tenant_id: str, k: int = 3):
    """전제: 세션 시작 시 테넌트를 고정해 retriever를 묶어 반환."""
    return vs.as_retriever(
        search_type='similarity',
        search_kwargs={'k': k, 'filter': {'tenant_id': {'$eq': tenant_id}}},
    )

retail = tenant_retriever('retail_bank')
sec = tenant_retriever('securities')

q = '수수료 구조 알려줘'
print('--- retail_bank ---')
for d in retail.invoke(q):
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["source"]}: {d.page_content[:60]}')
print('--- securities ---')
for d in sec.invoke(q):
    print(f'  {d.metadata["tenant_id"]}/{d.metadata["source"]}: {d.page_content[:60]}')

In [ ]:
# 교차 노출 테스트: retail 직원이 securities 문서를 조회할 수 있는가?
result = retail.invoke('ELS 중도상환 수수료')
only_retail = all(d.metadata['tenant_id'] == 'retail_bank' for d in result)
print(f'retail 테넌트만 반환되는가? {only_retail}')
for d in result:
    print(f'  tenant={d.metadata["tenant_id"]}  content={d.page_content[:40]}…')

### 실전 팁
- **콜렉션 분리**가 가능하다면 테넌트별 콜렉션 생성이 가장 안전. (인덱스/스토리지의 물리적 분리)
- 부득이 동일 콜렉션을 써야 한다면 Retriever 생성 시 필터를 하드코딩해 **우연히 필터를 빼는 실수**를 원천 차단.
- 감사 로그에는 사용자 `tenant_id`와 실제 필터 조건이 함께 남아야 한다.

## 4. Time-weighted Retriever

단순 유사도만 아니라 **"최근에 언급된/수정된" 문서에 가중치**를 줄 수 있다.

```
score = similarity + decay_rate ^ hours_since_last_accessed
```
약관이 개정되거나 공지사항이 갱신되는 금융 도메인에 적합.

In [ ]:
from langchain.retrievers import TimeWeightedVectorStoreRetriever
from datetime import datetime

# decay_rate: 0에 가까울수록 시간 가중치 영향이 큼, 1에 가까울수록 similarity만 사용
tw_retriever = TimeWeightedVectorStoreRetriever(
    vectorstore=vs,
    decay_rate=0.01,
    k=3,
)
# 날짜 메타를 last_accessed_at 으로 적재 (재인덱싱)
# Chroma는 datetime 객체를 메타데이터에 넣을 수 없으므로 ISO 문자열로 변환
from dateutil.parser import parse as dtparse
tw_docs = []
for d in docs:
    d_copy = d.copy()
    d_copy.metadata['last_accessed_at'] = dtparse(d.metadata['date']).isoformat()
    tw_docs.append(d_copy)
tw_retriever.add_documents(tw_docs)

print('Time-weighted 결과 (최근 문서에 조금 더 가중치):')
for d in tw_retriever.invoke('수수료'):
    print(f'  date={d.metadata["date"]}  {d.page_content[:60]}…')

## 정리

| 패턴 | 쓰는 상황 |
|---|---|
| **similarity** | 기본. 질의와 단순 유사한 문서가 필요할 때 |
| **MMR** | 답변에 다양한 정보(약관/FAQ/공지) 조합이 필요할 때 |
| **threshold** | 근거가 확실할 때만 답하고 싶을 때 (규정 내외 판단) |
| **metadata filter** | 부서/날짜/보안등급 등 사전 조건이 있을 때 |
| **tenant filter** | 금융권 사업부별 정보 격리 필수 |
| **time-weighted** | 갱신이 잦은 규정/공지 도메인 |

## 과제
1. **하이브리드 리트리버**: `BM25Retriever`와 `similarity_retriever`를 `EnsembleRetriever`로 엮어 가중치 0.4/0.6으로 실험해 보세요. (BM25 토크나이저는 `common.ko_tokenizer.tokenize_ko` 사용)
2. `TimeWeightedVectorStoreRetriever`의 `decay_rate`를 0.0001 / 0.01 / 0.1로 바꿔보며 랜킹이 어떻게 변하는지 관찰하세요.
3. 멀티테넌트 retriever를 함수로 만들고, `tenant_id`가 없는 호출에 대해 `ValueError`를 던지도록 **fail-safe**를 추가하세요.

## 더 읽어보기
- teddylee777/langchain-kr — [10-Retriever](https://github.com/teddylee777/langchain-kr/tree/main/10-Retriever)
- [LangChain Retrievers Concept](https://python.langchain.com/docs/concepts/retrievers/)
- [MMR 논문 (Carbonell & Goldstein, 1998)](https://dl.acm.org/doi/10.1145/290941.291025)